# 配套实践 15-02：用 ensemble 检查误差与可信区间

本练习训练五个概率动力学模型。每个模型不仅预测下一状态的均值，还预测数据本身的随机波动；不同模型之间的分歧则用于近似数据覆盖不足带来的不确定性。最后让模型从训练区域滚动到陌生区域，并同时检查误差、覆盖率和区间宽度。依赖：PyTorch、NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/15-long-horizon-imagination-and-uncertainty/" target="_blank">在新标签页返回课程正文</a>


In [ ]:
import numpy as np  # 生成动力学数据并计算区间统计量
import torch  # 训练概率神经网络并执行随机 rollout
from torch import nn  # 定义多层感知机与高斯输出头
import matplotlib.pyplot as plt  # 绘制数据覆盖、模型分歧和长期校准结果
torch.manual_seed(7)  # 固定 PyTorch 随机种子以便复现实验
np.random.seed(7)  # 固定 NumPy 随机种子以便复现实验
torch.set_num_threads(1)  # 限制 CPU 线程数以保持 Colab 运行稳定
plt.rcParams["figure.dpi"] = 120  # 提高笔记本内图像的清晰度
device = torch.device("cpu")  # 使用所有 Colab 运行时都支持的 CPU
print(f"运行设备：{device}；本练习将训练 5 个小型概率模型。")  # 告知读者当前运行配置


## 1. 构造具有两类不确定性的训练数据

真实系统接收一维状态 $x_t$ 和动作 $a_t$，再输出状态增量。动作越大，未建模的接触波动越明显，因此数据噪声随 $|a_t|$ 增大；这部分即使收集更多同分布样本也不会完全消失。训练样本只覆盖 $x,a\in[-1,1]$；真实系统在更远位置还会受到边界阻尼，但训练数据没有提供这部分证据。


In [ ]:
sample_count = 3500  # 设置训练样本数量
current_states = torch.empty(sample_count, 1).uniform_(-1.0, 1.0)  # 在有限状态范围内均匀采样
actions = torch.empty(sample_count, 1).uniform_(-1.0, 1.0)  # 在有限动作范围内均匀采样
mean_deltas = 0.25 * actions + 0.08 * torch.sin(3.0 * current_states)  # 计算真实动力学的平均状态增量
noise_stds = 0.04 + 0.04 * torch.abs(actions)  # 让大动作对应更强的环境随机性
observed_deltas = mean_deltas + noise_stds * torch.randn_like(mean_deltas)  # 加入无法由输入消除的随机波动
figure, axes = plt.subplots(1, 2, figsize=(10.4, 3.8), constrained_layout=True)  # 建立数据覆盖与噪声规律两个子图
scatter = axes[0].scatter(current_states.numpy(), actions.numpy(), c=observed_deltas.numpy(), s=8, alpha=0.42, cmap="coolwarm")  # 显示训练输入覆盖及对应增量
axes[0].set(xlabel="State x", ylabel="Action a", title="Training coverage")  # 标注训练覆盖图
figure.colorbar(scatter, ax=axes[0], label="Observed delta")  # 解释散点颜色含义
action_grid = np.linspace(-1.0, 1.0, 200)  # 建立用于显示噪声规律的连续动作网格
axes[1].plot(action_grid, 0.04 + 0.04 * np.abs(action_grid), color="#245b78", linewidth=2.4)  # 绘制真实随机噪声标准差
axes[1].fill_between(action_grid, 0.0, 0.04 + 0.04 * np.abs(action_grid), color="#91bfd1", alpha=0.35)  # 突出动作相关噪声区域
axes[1].set(xlabel="Action a", ylabel="Noise standard deviation", title="Irreducible variation", ylim=(0.0, 0.09))  # 标注随机性规律图
plt.show()  # 在笔记本中输出训练数据图


**怎样理解结果：** 左图中的点只覆盖中央方形区域，模型不能仅凭这批数据确认更远状态的动力学；右图表示同一状态和动作也可能产生略有不同的结果，这种不可约波动属于 aleatoric uncertainty。后面若分布外区域的模型成员开始彼此分歧，则主要反映 epistemic uncertainty。


## 2. 训练五个概率动力学模型

每个网络输出增量均值 $\mu_	heta(x,a)$ 与对数标准差 $\log\sigma_	heta(x,a)$，使用高斯负对数似然训练。五个成员采用不同初始化和 bootstrap 数据；因此它们共享任务，却没有完全相同的训练经历。


In [ ]:
class ProbabilisticDynamics(nn.Module):  # 定义同时预测均值和标准差的动力学模型
    def __init__(self):  # 初始化网络层
        super().__init__()  # 调用神经网络基类初始化逻辑
        self.network = nn.Sequential(nn.Linear(2, 48), nn.Tanh(), nn.Linear(48, 48), nn.Tanh(), nn.Linear(48, 2))  # 用两层非线性网络输出两个高斯参数
    def forward(self, state, action):  # 根据状态和动作执行一次前向计算
        raw_output = self.network(torch.cat([state, action], dim=1))  # 拼接条件并计算网络原始输出
        mean_delta = raw_output[:, :1]  # 取第一维作为状态增量均值
        log_std = -4.0 + 4.0 * torch.sigmoid(raw_output[:, 1:])  # 用平滑映射限制标准差范围并保留有效梯度
        return mean_delta, log_std  # 返回可分别解释的均值与对数标准差
ensemble_size = 5  # 设置 ensemble 成员数量
epoch_count = 50  # 设置每个小模型的训练轮数
batch_size = 250  # 设置每次参数更新使用的样本数量
ensemble = []  # 保存训练完成的模型成员
loss_histories = []  # 保存每个成员的逐轮训练损失
for member_index in range(ensemble_size):  # 依次训练具有不同数据重采样的成员
    model = ProbabilisticDynamics().to(device)  # 创建当前概率动力学模型
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)  # 使用 Adam 更新模型参数
    bootstrap_indices = torch.randint(sample_count, (sample_count,))  # 有放回抽样形成当前成员的数据集
    bootstrap_states = current_states[bootstrap_indices]  # 读取当前成员的状态样本
    bootstrap_actions = actions[bootstrap_indices]  # 读取当前成员的动作样本
    bootstrap_targets = observed_deltas[bootstrap_indices]  # 读取当前成员的增量目标
    member_losses = []  # 记录当前成员每轮的平均损失
    for epoch_index in range(epoch_count):  # 多轮遍历 bootstrap 数据以学习概率分布
        shuffled_indices = torch.randperm(sample_count)  # 每轮随机改变样本顺序
        epoch_losses = []  # 收集当前轮各批次损失
        for batch_start in range(0, sample_count, batch_size):  # 按小批量进行参数更新
            batch_indices = shuffled_indices[batch_start:batch_start + batch_size]  # 选出当前批次索引
            predicted_mean, predicted_log_std = model(bootstrap_states[batch_indices], bootstrap_actions[batch_indices])  # 预测当前批次的均值和标准差
            predicted_std = predicted_log_std.exp()  # 将对数标准差还原为正数标准差
            normalized_error = (bootstrap_targets[batch_indices] - predicted_mean) / predicted_std  # 用预测尺度标准化增量误差
            loss = (0.5 * normalized_error.square() + predicted_log_std).mean()  # 计算省略常数项的高斯负对数似然
            optimizer.zero_grad()  # 清空上一次更新遗留的梯度
            loss.backward()  # 反向传播当前概率损失
            optimizer.step()  # 根据梯度更新当前模型参数
            epoch_losses.append(loss.item())  # 保存当前批次损失用于汇总
        member_losses.append(float(np.mean(epoch_losses)))  # 记录当前训练轮的平均损失
    ensemble.append(model.eval())  # 将训练完成的成员切换到推理模式并保存
    loss_histories.append(member_losses)  # 保存当前成员的完整损失曲线
figure, ax = plt.subplots(figsize=(8.4, 3.8), constrained_layout=True)  # 建立 ensemble 训练曲线图
for member_index, member_losses in enumerate(loss_histories):  # 逐个绘制五个成员的训练过程
    ax.plot(member_losses, linewidth=1.6, alpha=0.82, label=f"Member {member_index + 1}")  # 显示当前成员的负对数似然变化
ax.set(xlabel="Epoch", ylabel="Gaussian NLL (without constant)", title="Five bootstrap members learn the same dynamics")  # 标注训练曲线坐标和标题
ax.legend(ncol=5, fontsize=8)  # 用紧凑图例区分 ensemble 成员
ax.grid(alpha=0.2)  # 添加浅色网格帮助读取趋势
plt.show()  # 在笔记本中输出训练曲线


**怎样理解结果：** 五条曲线都下降并进入相近区间，说明各成员已学到训练区域内的规律。损失为负并不是错误：这里省略了高斯密度的常数项，而且较小的合理标准差会产生负的 $\log\sigma$。成员都能拟合训练集也不意味着它们在陌生区域拥有相同判断。


## 3. 把总不确定性拆成两部分

固定动作 $a=0.5$，让模型评估更宽的状态范围。成员自身输出方差的平均值近似 aleatoric 部分，成员均值之间的方差近似 epistemic 部分；两者相加得到总预测方差。


In [ ]:
state_grid = torch.linspace(-2.5, 2.5, 301).unsqueeze(1)  # 建立横跨训练内外区域的状态网格
fixed_actions = torch.full_like(state_grid, 0.5)  # 在所有状态处使用相同动作以便比较
member_next_means = []  # 收集各成员预测的下一状态均值
member_next_stds = []  # 收集各成员预测的随机标准差
with torch.no_grad():  # 关闭梯度以执行高效推理
    for model in ensemble:  # 让每个成员独立评估完整网格
        predicted_delta, predicted_log_std = model(state_grid, fixed_actions)  # 预测状态增量分布参数
        member_next_means.append((state_grid + predicted_delta).squeeze(1).numpy())  # 将增量均值换算为下一状态均值
        member_next_stds.append(predicted_log_std.exp().squeeze(1).numpy())  # 保存当前成员的随机标准差
member_next_means = np.asarray(member_next_means)  # 将五组均值整理为成员乘网格的数组
member_next_stds = np.asarray(member_next_stds)  # 将五组标准差整理为成员乘网格的数组
grid_values = state_grid.squeeze(1).numpy()  # 将状态网格转换为绘图数组
ensemble_mean = member_next_means.mean(axis=0)  # 计算 ensemble 下一状态平均预测
aleatoric_std = np.sqrt(np.mean(member_next_stds ** 2, axis=0))  # 汇总成员内部方差得到随机不确定性
epistemic_std = member_next_means.std(axis=0)  # 用成员均值分歧近似模型不确定性
total_std = np.sqrt(aleatoric_std ** 2 + epistemic_std ** 2)  # 根据全方差公式合成总标准差
boundary_drag = -0.12 * np.sign(grid_values) * np.maximum(np.abs(grid_values) - 1.0, 0.0) ** 2  # 计算训练范围外才出现的真实边界阻尼
true_next_mean = grid_values + 0.25 * 0.5 + 0.08 * np.sin(3.0 * grid_values) + boundary_drag  # 计算包含未知边界效应的真实平均动力学
inside_mask = np.abs(grid_values) <= 1.0  # 标记训练状态覆盖区域
outside_mask = np.abs(grid_values) >= 1.8  # 标记明显分布外的状态区域
figure, axes = plt.subplots(1, 2, figsize=(11.0, 4.1), constrained_layout=True)  # 建立预测区间和不确定性分解两个子图
axes[0].axvspan(-1.0, 1.0, color="#dce9df", alpha=0.7, label="Training range")  # 用背景色标明训练状态范围
axes[0].fill_between(grid_values, ensemble_mean - 1.64 * total_std, ensemble_mean + 1.64 * total_std, color="#d49a57", alpha=0.28, label="Model 90% interval")  # 绘制近似百分之九十预测区间
axes[0].plot(grid_values, true_next_mean, color="#27364a", linewidth=2.1, label="True mean")  # 绘制真实平均下一状态
axes[0].plot(grid_values, ensemble_mean, color="#b35c22", linewidth=2.0, linestyle="--", label="Ensemble mean")  # 绘制 ensemble 平均预测
axes[0].set(xlabel="Current state x", ylabel="Next state", title="Prediction inside and outside training support")  # 标注下一状态预测图
axes[0].legend(fontsize=8)  # 显示训练范围和预测曲线图例
axes[0].grid(alpha=0.18)  # 添加浅色网格帮助比较曲线
axes[1].axvspan(-1.0, 1.0, color="#dce9df", alpha=0.7, label="Training range")  # 在分解图中重复标明训练范围
axes[1].plot(grid_values, aleatoric_std, color="#367c8d", linewidth=2.1, label="Aleatoric")  # 绘制成员内部随机不确定性
axes[1].plot(grid_values, epistemic_std, color="#a44848", linewidth=2.1, label="Epistemic proxy")  # 绘制成员之间的模型分歧
axes[1].set(xlabel="Current state x", ylabel="Standard deviation", title="Two sources of uncertainty")  # 标注不确定性分解图
axes[1].legend(fontsize=8)  # 显示两类不确定性图例
axes[1].grid(alpha=0.18)  # 添加浅色网格帮助读取变化
plt.show()  # 在笔记本中输出单步不确定性图
inside_epistemic = epistemic_std[np.abs(grid_values) < 0.8].mean()  # 汇总训练内部的成员分歧
outside_epistemic = epistemic_std[outside_mask].mean()  # 汇总明显分布外区域的成员分歧
print(f"训练内部平均 epistemic std：{inside_epistemic:.4f}")  # 报告训练区域内部的模型不确定性
print(f"远离训练区域平均 epistemic std：{outside_epistemic:.4f}")  # 报告分布外区域的模型不确定性


**怎样理解结果：** 绿色区域是模型真正见过的状态范围。在区域内部，aleatoric 估计接近固定动作对应的真实噪声，epistemic 分歧较低；离开数据覆盖后，分歧通常升高，概率头本身也会外推。真实曲线还受到模型从未见过的边界阻尼影响。阴影带同时包含两部分不确定性，却仍不是安全证明：不同网络可能在分布外共同犯错。


## 4. 随 horizon 检查误差、覆盖率和区间宽度

从训练区域内的 $x=-0.5$ 出发，持续执行 $a=0.55$。轨迹会逐渐走出训练范围。我们分别从真实系统和 ensemble 采样大量未来，再按 horizon 比较平均误差、90% 区间覆盖率与区间宽度。


In [ ]:
horizon_count = 20  # 设置自由 rollout 的未来步数
true_rollout_count = 500  # 设置真实随机轨迹的采样数量
particles_per_member = 120  # 设置每个模型成员传播的粒子数量
start_state = -0.5  # 从训练分布内部选择初始状态
rollout_action = 0.55  # 使用持续正动作把系统逐步推向分布外区域
random_generator = np.random.default_rng(10)  # 建立可复现的真实系统随机数生成器
true_rollouts = np.zeros((true_rollout_count, horizon_count + 1))  # 为全部真实轨迹预留存储空间
true_rollouts[:, 0] = start_state  # 写入所有真实轨迹的相同初始状态
for horizon_index in range(horizon_count):  # 逐步推进真实随机动力学
    previous_states = true_rollouts[:, horizon_index]  # 读取当前 horizon 的真实状态
    rollout_boundary_drag = -0.12 * np.sign(previous_states) * np.maximum(np.abs(previous_states) - 1.0, 0.0) ** 2  # 计算离开训练区域后的真实边界阻尼
    true_mean_deltas = 0.25 * rollout_action + 0.08 * np.sin(3.0 * previous_states) + rollout_boundary_drag  # 计算包含未知边界效应的真实平均状态增量
    true_noise_std = 0.04 + 0.04 * abs(rollout_action)  # 计算当前固定动作下的随机标准差
    true_rollouts[:, horizon_index + 1] = previous_states + true_mean_deltas + true_noise_std * random_generator.standard_normal(true_rollout_count)  # 采样并保存下一真实状态
model_particles = np.full((ensemble_size, particles_per_member, horizon_count + 1), start_state, dtype=np.float32)  # 为各成员的随机粒子预留空间
for horizon_index in range(horizon_count):  # 让 ensemble 逐步自由预测未来
    for member_index, model in enumerate(ensemble):  # 使用每个成员传播自己的随机粒子
        previous_particles = torch.tensor(model_particles[member_index, :, horizon_index, None])  # 读取当前成员的上一时刻粒子
        particle_actions = torch.full_like(previous_particles, rollout_action)  # 为所有粒子配置相同候选动作
        with torch.no_grad():  # 关闭梯度以执行随机模型 rollout
            predicted_delta, predicted_log_std = model(previous_particles, particle_actions)  # 预测每个粒子的下一增量分布
        sampled_next_particles = previous_particles + predicted_delta + predicted_log_std.exp() * torch.randn_like(previous_particles)  # 从当前成员的预测分布采样下一状态
        model_particles[member_index, :, horizon_index + 1] = sampled_next_particles.squeeze(1).numpy()  # 保存下一 horizon 的全部模型粒子
flat_model_rollouts = model_particles.reshape(-1, horizon_count + 1)  # 合并成员和粒子维度形成预测样本集
horizons = np.arange(horizon_count + 1)  # 建立从零到最大步数的 horizon 横轴
model_mean_by_horizon = flat_model_rollouts.mean(axis=0)  # 计算每个 horizon 的模型平均状态
model_lower_by_horizon = np.quantile(flat_model_rollouts, 0.05, axis=0)  # 计算每个 horizon 的预测区间下界
model_upper_by_horizon = np.quantile(flat_model_rollouts, 0.95, axis=0)  # 计算每个 horizon 的预测区间上界
true_mean_by_horizon = true_rollouts.mean(axis=0)  # 计算每个 horizon 的真实平均状态
mean_error_by_horizon = np.abs(model_mean_by_horizon - true_mean_by_horizon)  # 计算模型平均值与真实平均值的绝对差
coverage_by_horizon = ((true_rollouts >= model_lower_by_horizon) & (true_rollouts <= model_upper_by_horizon)).mean(axis=0)  # 统计真实样本落入模型区间的比例
width_by_horizon = model_upper_by_horizon - model_lower_by_horizon  # 计算模型预测区间宽度
figure, axes = plt.subplots(1, 2, figsize=(11.4, 4.2), constrained_layout=True)  # 建立长期轨迹和诊断指标两个子图
for rollout_index in range(0, true_rollout_count, 50):  # 抽取少量真实轨迹避免图像过密
    axes[0].plot(horizons, true_rollouts[rollout_index], color="#8a969e", linewidth=0.8, alpha=0.25)  # 显示真实随机未来的多样性
axes[0].fill_between(horizons, model_lower_by_horizon, model_upper_by_horizon, color="#d49a57", alpha=0.3, label="Model 90% interval")  # 绘制模型长期预测区间
axes[0].plot(horizons, true_mean_by_horizon, color="#263746", linewidth=2.2, label="True mean")  # 绘制真实平均轨迹
axes[0].plot(horizons, model_mean_by_horizon, color="#b35c22", linewidth=2.1, linestyle="--", label="Model mean")  # 绘制模型平均轨迹
axes[0].axhspan(-1.0, 1.0, color="#dce9df", alpha=0.38, label="Training state range")  # 标明训练状态覆盖范围
axes[0].set(xlabel="Rollout horizon", ylabel="State x", title="Uncertainty grows during free rollout")  # 标注长期轨迹图
axes[0].legend(fontsize=8)  # 显示长期轨迹图例
axes[0].grid(alpha=0.18)  # 添加浅色网格帮助定位 horizon
coverage_axis = axes[1].twinx()  # 创建独立纵轴以显示零到一之间的覆盖率
axes[1].plot(horizons, mean_error_by_horizon, color="#a44848", linewidth=2.1, label="Mean error")  # 绘制随 horizon 变化的均值误差
axes[1].plot(horizons, width_by_horizon, color="#367c8d", linewidth=2.1, label="Interval width")  # 绘制随 horizon 变化的预测区间宽度
coverage_axis.plot(horizons, coverage_by_horizon, color="#557345", linewidth=2.1, linestyle="--", label="Coverage")  # 绘制百分之九十区间的实际覆盖率
coverage_axis.axhline(0.9, color="#557345", linewidth=1.0, alpha=0.55)  # 标出目标覆盖率参考线
axes[1].set(xlabel="Rollout horizon", ylabel="Error or interval width", title="Calibration must include sharpness")  # 标注误差和区间宽度坐标
coverage_axis.set(ylabel="Coverage", ylim=(0.0, 1.05))  # 标注覆盖率坐标并限制合理范围
left_lines, left_labels = axes[1].get_legend_handles_labels()  # 读取左纵轴图例项目
right_lines, right_labels = coverage_axis.get_legend_handles_labels()  # 读取右纵轴图例项目
axes[1].legend(left_lines + right_lines, left_labels + right_labels, fontsize=8, loc="upper left")  # 合并两个纵轴的图例
axes[1].grid(alpha=0.18)  # 添加浅色网格帮助比较指标
plt.show()  # 在笔记本中输出长期不确定性诊断图
print(f"horizon 1：均值误差 {mean_error_by_horizon[1]:.3f}，覆盖率 {coverage_by_horizon[1]:.1%}，区间宽度 {width_by_horizon[1]:.3f}")  # 报告短期预测的三项诊断
print(f"horizon {horizon_count}：均值误差 {mean_error_by_horizon[-1]:.3f}，覆盖率 {coverage_by_horizon[-1]:.1%}，区间宽度 {width_by_horizon[-1]:.3f}")  # 报告长期预测的三项诊断


**怎样理解结果：** horizon 0 是已知起点。随着模型反复使用自己的采样结果，均值误差和区间宽度都会改变；进入训练范围之外后，这些变化通常更明显。覆盖率接近或高于 90% 不能单独证明模型优秀，因为非常宽的区间也容易覆盖真实结果。可信模型应在保持合理覆盖率的同时尽量给出窄而有区分力的区间。

**本练习的结论：** 单模型概率头可描述输入条件下仍然存在的随机性，bootstrap ensemble 的成员分歧可作为模型未知的实用近似。规划器必须同时读取预测均值、不确定性和 horizon：当误差或区间宽度超过任务可接受范围时，应缩短规划 horizon、重新观测，或拒绝执行高风险候选。
